In [ ]:
#Requirements Installation 

%pip install openpyxl

In [ ]:
#Imports

from pyspark.sql import SparkSession
from datetime import datetime
import importlib.util
import sys
import os
import yaml
 
spark = SparkSession.builder.getOrCreate()

In [ ]:
#Path's Definitions

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
notebook_path = ctx.notebookPath().get()
 
NOTEBOOK_DIR = f"/Workspace{notebook_path.rsplit('/', 1)[0]}"
SCRIPTS_DIR = os.path.abspath(f"{NOTEBOOK_DIR}/../scripts")

INGESTION_DIR = os.path.abspath(f"{NOTEBOOK_DIR}/../ingestion_framework") 
 
print("Notebook Directory:", NOTEBOOK_DIR)
print("Scripts Directory :", SCRIPTS_DIR)
print("ingestion Directory :", INGESTION_DIR)

In [ ]:
#variables Definitions

dbutils.widgets.text("raw_bucket", "")
dbutils.widgets.text("stg_bucket", "")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("silver_schema", "")
dbutils.widgets.text("gold_schema", "")
dbutils.widgets.text("metadata_catalog", "")
dbutils.widgets.text("registry_schema", "")
dbutils.widgets.text("metadata_schema", "")
dbutils.widgets.text("run_date", "")
dbutils.widgets.text("environment", "")
dbutils.widgets.text("source_bucket", "")
dbutils.widgets.text("simulation_type", "")


run_date = dbutils.widgets.get("run_date").strip() or datetime.now().strftime("%Y%m%d")

spark.conf.set("quality.environment", dbutils.widgets.get("environment")) 
spark.conf.set("quality.raw_bucket", dbutils.widgets.get("raw_bucket"))
spark.conf.set("quality.stg_bucket", dbutils.widgets.get("stg_bucket"))
spark.conf.set("quality.CATALOG", dbutils.widgets.get("catalog"))
spark.conf.set("quality.BRONZE_SCHEMA", dbutils.widgets.get("bronze_schema"))
spark.conf.set("quality.SILVER_SCHEMA", dbutils.widgets.get("silver_schema"))
spark.conf.set("quality.GOLD_SCHEMA", dbutils.widgets.get("gold_schema"))
spark.conf.set("quality.simulation_type", dbutils.widgets.get("simulation_type"))


spark.conf.set("quality.METADATA_CATALOG", dbutils.widgets.get("metadata_catalog"))
spark.conf.set("quality.METADATA_SCHEMA", dbutils.widgets.get("metadata_schema"))
spark.conf.set("quality.REGISTRY_SCHEMA", dbutils.widgets.get("registry_schema"))
spark.conf.set("quality.source_bucket", dbutils.widgets.get("source_bucket"))
spark.conf.set("quality.YML_CONFIG_PATH", f"{INGESTION_DIR}/configs/computer_periodic_review_config.yml")
spark.conf.set("quality.SCHEMA_REGISTRY_PATH", f"{INGESTION_DIR}/ingestion_engine/schema_registry.py")
spark.conf.set("quality.RUN_DATE", run_date)


METADATA_CATALOG = spark.conf.get("quality.METADATA_CATALOG")
METADATA_SCHEMA = spark.conf.get("quality.METADATA_SCHEMA")
REGISTRY_SCHEMA = spark.conf.get("quality.REGISTRY_SCHEMA")
source_bucket = spark.conf.get("quality.source_bucket")
environment = spark.conf.get("quality.environment")
simulation_type = spark.conf.get("quality.simulation_type")

print("**quality Simulation Type** =", simulation_type)
print("RUN_DATE =", run_date)

In [ ]:
#Import Helper Function
 
def import_from_path(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
 
    # Inject dbutils into the module
    mod.dbutils = dbutils
 
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

In [ ]:
config_path = f"{INGESTION_DIR}/configs/computer_periodic_review_config.yml"
with open(config_path, "r") as f:
    quality_config = yaml.safe_load(f)
    
schema_reg = import_from_path(
    "schema_registry",
    f"{INGESTION_DIR}/ingestion_engine/schema_registry.py"
)

In [ ]:
METADATA_CATALOG = spark.conf.get("quality.METADATA_CATALOG")
REGISTRY_SCHEMA  = spark.conf.get("quality.REGISTRY_SCHEMA")
METADATA_SCHEMA  = spark.conf.get("quality.METADATA_SCHEMA")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.dataset_registry (
  dataset_id STRING NOT NULL,
  domain_name STRING,
  data_product_name STRING,
  dataset_name STRING,
  dataset_version STRING,
  frequency STRING,
  owner_team STRING,
  owner_email STRING,
  criticality STRING,
  contains_pii BOOLEAN,
  data_classification STRING,
  lifecycle_status STRING,
  retention_days INT,
  created_at TIMESTAMP,
  is_active BOOLEAN
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.ingestion_config (
  config_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  environment STRING,
  source_bucket STRING,
  source_path STRING,
  file_format STRING,
  delimiter STRING,
  file_encoding STRING,
  load_type STRING,
  file_name STRING,
  row_tag STRING,
  ingestion_mode STRING,
  primary_keys STRING,
  watermark_column STRING,
  checkpoint_location STRING,
  schema_location STRING,
  schema_strategy STRING,
  optimize_write BOOLEAN,
  auto_compact BOOLEAN,
  is_active BOOLEAN
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.dataset_tags (
  tag_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  tag_key STRING,
  tag_value STRING,
  created_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.dataset_dependencies (
  dependency_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  depends_on_dataset_id STRING,
  dependency_type STRING,
  created_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.ingestion_runtime_state (
  runtime_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  environment STRING,
  last_run_status STRING,
  records_ingested BIGINT,
  files_processed INT,
  last_run_start_time TIMESTAMP,
  last_run_end_time TIMESTAMP,
  failure_reason STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.dq_sla_config (
  dq_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  dq_enabled BOOLEAN,
  rule_set_name STRING,
  row_count_min BIGINT,
  row_count_max BIGINT,
  freshness_minutes INT,
  null_threshold_pct DOUBLE,
  duplicate_threshold_pct DOUBLE,
  anomaly_detection_enabled BOOLEAN,
  fail_action STRING,
  alert_channel STRING,
  escalation_contact STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.schema_registry (
  dataset_id STRING NOT NULL,
  schema_json STRING NOT NULL,
  version INT,
  is_active BOOLEAN,
  created_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"CREATE VOLUME IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.schemas")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.checkpoints")

In [ ]:
#Run metadata loader

metadata_loader = import_from_path(
    "metadata_loader",
    f"{INGESTION_DIR}/metadata_service/metadata_loader.py"
)

metadata_loader.main()

In [ ]:
# Autoloader engine

RAW_BUCKET = spark.conf.get("quality.raw_bucket")
autoloader_engine = import_from_path(
    "autoloader_engine",
    f"{INGESTION_DIR}/ingestion_engine/autoloader_engine.py"
)
autoloader_engine.main(simulation_type)